# W14 · 01 기술자 → 학습 표 → XGBoost 그룹 5-fold

실행 = week14 폴더의 Claude Code 에 「`@notebooks/01_features_xgb.ipynb` 를 실행하고 결과를 요약해」 → week14 폴더에서 `python ../run_nb.py notebooks/01_features_xgb.ipynb` (직접 보려면 `jupyter lab` 에서 모두 실행) → `out/mol_descriptors.csv` · `out/features.csv` · `out/train_metrics.json`. 설명은 덱 4 ~ 15장 · `../README.md`.
격자 60회까지 약 2분(1코어). 빨리 보려면 3번 셀의 `FAST = True`.

In [ ]:
# 로컬 · Claude Code 가 기본: rdkit · xgboost · shap · scikit-learn 은 mdai 에 포함(../../environment.yml) · 설치 불필요
# (선택) Colab 이면 아래 줄의 주석을 푼다 (1회 · 약 1분)
# !pip -q install rdkit xgboost shap scikit-learn pandas
import subprocess, sys, json, os, pathlib
if pathlib.Path('scripts').is_dir() and pathlib.Path('notebooks').is_dir():
    os.chdir('notebooks')   # run_nb.py(week14 에서 시작) → 이 노트북의 경로 기준(notebooks/)으로 · JupyterLab 은 그대로
import pandas as pd
def run(script, *args):
    r = subprocess.run([sys.executable, f'scripts/{script}', *args], cwd='..', capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-800:])

## 1. 분자 32개 → 기술자 14열 (덱 5장 · P14-1)
내 분자를 넣으려면 `../data/sara_molecules.csv` 끝에 SMILES 1행을 추가하고 이 셀을 다시 실행한다.

In [ ]:
run('descriptors.py')
md = pd.read_csv('../out/mol_descriptors.csv')
md[md.mol_id.str.contains('AS-Thio')][['mol_id', 'n_heavy', 'O_frac', 'aro_frac', 'logP', 'TPSA', 'Rg', 'aspect']]

## 2. 바인더 9행 · 학습 표 672행 · 54그룹 (덱 6 · 10장)

In [ ]:
run('binder_features.py')
print(json.load(open('../out/features_log.json')))
pd.read_csv('../out/binder_features.csv')[['binder_type', 'aging_state', 'b_O_frac', 'b_logP', 'b_aro_frac', 'b_TPSA']]

## 3. XGBoost 4표적 × 특징 세트 3 · Ridge · 무작위 ↔ 그룹 · 격자 (덱 11 ~ 15장 · P14-2)

In [ ]:
FAST = False   # True = 격자 60회 생략 (이전 hparam_grid.json 재사용)
run('train_xgb.py', *(['--fast'] if FAST else []))
m = json.load(open('../out/train_metrics.json'))
rows = []
for t in ('density', 'CED', 'bulk', 'E_inter'):
    s = m[t]['sets']
    rows.append({'표적': t, 'n': m[t]['n'], '온도만': s['T']['R2'], '+바인더': s['T+binder']['R2'],
                 '+첨가제': s['T+binder+additive']['R2'], 'Ridge': m[t]['ridge_full'],
                 'RMSE': f"{s['T+binder+additive']['RMSE']:.4g} {m[t]['unit']}"})
pd.DataFrame(rows)

In [ ]:
print('무작위 ↔ 그룹', m['split_compare'])
print('부스팅 시연', m['boosting_demo'])
print('격자', m.get('hparam'))

**확인** 밀도 +첨가제 R² 0.975 · 무작위 0.989 ↔ 그룹 0.975 · 격자 최고 800 · 2 · 0.03 → 0.986 (`../expected_output/train_metrics.json` 과 비교)

**바꿔 보기** `scripts/` 는 기준 코드라 고치지 않는다. Claude Code 에 「`scripts/common.py` 의 `PARAMS` 에서 `max_depth` 만 5 → 2 로 바꾼 변형을 `out/` 아래 새 스크립트로 만들어 돌리고, 결과를 다른 이름으로 저장해 `out/train_metrics.json` 과 R² 를 비교해」.